<a href="https://colab.research.google.com/github/lokesh9460/super-resolution-of-sentinal-2-images-using-GAN/blob/master/Copy_of_Copy_of_Sentinel_Cartosat_EDSR_With_ESRGAN_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
d=[]
while(1):
  d.append('1')

KeyboardInterrupt: 

In [ ]:
# Importing the libraries

import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import cv2

from skimage.io import imsave

import tensorflow
from tensorflow.keras.layers import Input, Dense , Add , Lambda , Concatenate, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation, LeakyReLU
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.layers import Conv2D, UpSampling2D
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.optimizers import Adam

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path='/content/drive/My Drive/Super_Resolution/EDSR_ESRGAN/'
glob.glob( os.path.join(path,'Train_Images','*.jpg'))

['/content/drive/My Drive/Super_Resolution/EDSR_ESRGAN/Train_Images/sentinal.jpg',
 '/content/drive/My Drive/Super_Resolution/EDSR_ESRGAN/Train_Images/cartosat.jpg']

In [ ]:
# Loading the training images and preprocessing Step

def splitImg(image,dim):
    ''' Takes input image and splits it into multiple images of dimension dim
    Outputs an array consisting of smaller images in order '''

    X=image.shape[0]/dim[0]
    Y=image.shape[1]/dim[1]
    no_images=X*Y
    #print("No. of images:",no_images)
    #print("Shape of image:",image.shape)
    images=[]
    i=0;j=0
    count=0
    while i <image.shape[0]:
        j=0
        while j<image.shape[1]:
            img=image[i:i+dim[0],j:j+dim[1],:]
            images.append(img)
            #count=count+1
            #print("Count: ",count)
            j=j+dim[1]
        i=i+dim[0]
    return(np.array(images) , no_images)




In [ ]:
def dataPreProcessing(dimHR,dimLR):
  ''' Takes dim as input which is the dimension of required training images to be obtained from the satellite image
      Outputs two arrays of high_resolution and low_resloution images
  '''

  path='/content/drive/My Drive/Super_Resolution/EDSR_ESRGAN/'
  images = glob.glob( os.path.join(path,'Train_Images', '*.jpg'))

  low_resolution_image=cv2.imread(images[1])
  high_resolution_image=cv2.imread(images[0])

  print('Dimension of HR Cartosat Image: ',high_resolution_image.shape)
  print('Dimension of LR Sentinel Image: ',low_resolution_image.shape)

  train_hr , no_images = splitImg(high_resolution_image,dimHR)
  train_lr , no_images= splitImg(low_resolution_image,dimLR)

  print("Total No. of Images: ",int(no_images))

  train_hr=train_hr.astype('float64')
  train_lr=train_lr.astype('float64')

  return(train_hr , train_lr , int(no_images))

In [ ]:
# Function to randomly sample images from trainig set

def sampleImages(batch_size,train_hr,train_lr,no_images):
  # Generates a random batch of size batch_size


  random_batch = np.random.choice(no_images, batch_size , replace=False)

  high_resolution_images = np.array([train_hr[i] for i in random_batch])
  low_resolution_images = np.array([train_lr[i] for i in random_batch])
  return(high_resolution_images , low_resolution_images)

In [ ]:
def stichImages(images,dim):
  '''Takes images array and dimension to output image as input
  Outputs a stiched image of dimension dim '''


  out=np.zeros((dim[0],dim[1],3))
  X=images[0].shape[0];Y=images[0].shape[1]

  (i,j,k)=(0,0,0)
  while i < dim[0]:
      j=0
      while j < dim[1]:
          out[i:i+X,j:j+Y,:] = images[k]
          k=k+1
          j=j+Y
      i=i+X
  return(out)

In [ ]:
def save_images(low_resolution_image, original_image, generated_image,path):
    """
    Save low-resolution, high-resolution(original) and
    generated high-resolution images in a single image
    """

    fig = plt.figure()
    ax = fig.add_subplot(1, 3, 1)
    ax.imshow(low_resolution_image)
    ax.axis("off")
    ax.set_title("LR")

    ax = fig.add_subplot(1, 3, 2)
    ax.imshow(original_image)
    ax.axis("off")
    ax.set_title("Original HR")

    ax = fig.add_subplot(1, 3, 3)
    ax.imshow(generated_image)
    ax.axis("off")
    ax.set_title("Generated SR")

    plt.savefig(path)

In [ ]:
def build_model():
        """
        Build the generator network according to description in the paper.
        :return: the compiled model
        """

        def dense_block(input):
            x1 = Conv2D(64, kernel_size=3, strides=1, padding='same')(input)
            x1 = LeakyReLU(0.2)(x1)
            x1 = Concatenate()([input, x1])
            x1 = Dropout(0.2)(x1)

            x2 = Conv2D(64, kernel_size=3, strides=1, padding='same')(x1)
            x2 = LeakyReLU(0.2)(x2)
            x2 = Concatenate()([input, x1, x2])
            x2 = Dropout(0.2)(x2)


            x3 = Conv2D(64, kernel_size=3, strides=1, padding='same')(x2)
            x3 = LeakyReLU(0.2)(x3)
            x3 = Concatenate()([input, x1, x2, x3])
            x3 = Dropout(0.2)(x3)

            x4 = Conv2D(64, kernel_size=3, strides=1, padding='same')(x3)
            x4 = LeakyReLU(0.2)(x4)
            x4 = Concatenate()([input, x1, x2, x3, x4])
            x4 = Dropout(0.2)(x4)

            x5 = Conv2D(64, kernel_size=3, strides=1, padding='same')(x4)
            x5 = Lambda(lambda x: x * 0.2)(x5)
            x = Add()([x5, input])
            return x

        def RRDB(input):
            x = dense_block(input)
            x = dense_block(x)
            x = dense_block(x)
            x = Lambda(lambda x: x * 0.2)(x)
            out = Add()([x, input])
            return out

        def upsample(x, scale, num_filters):
          def upsample_1(x, factor, **kwargs):
            x = Conv2D(num_filters * (factor ** 2), 3, padding='same', **kwargs)(x)
            return Lambda(pixel_shuffle(scale=factor))(x)

          if scale == 2:
              x = upsample_1(x, 2, name='conv2d_1_scale_2')
          elif scale == 3:
              x = upsample_1(x, 3, name='conv2d_1_scale_3')
          elif scale == 4:
              x = upsample_1(x, 2, name='conv2d_1_scale_2')
              x = upsample_1(x, 2, name='conv2d_2_scale_2')

          return x


        def pixel_shuffle(scale):
          return lambda x: tensorflow.nn.depth_to_space(x, scale)

        # Input low resolution image
        lr_input = Input(shape=(100, 100, 3))

        # Pre-residual
        x_start = Conv2D(64, kernel_size=3, strides=1, padding='same')(lr_input)
        x_start = LeakyReLU(0.2)(x_start)

        # 10 Residual-in-Residual Dense Block
        x = RRDB(x_start)
        for i in range(9):
          x=RRDB(x)

        # Post-residual block
        x = Conv2D(64, kernel_size=3, strides=1, padding='same')(x)
        x = Lambda(lambda x: x * 0.2)(x)
        x = Add()([x, x_start])

        # Upsampling depending on factor
        x = upsample(x, scale = 2 , num_filters=64)

        x = Conv2D(64, kernel_size=3, strides=1, padding='same')(x)
        x = LeakyReLU(0.2)(x)
        hr_output = Conv2D(3, kernel_size=3, strides=1, padding='same', activation='tanh')(x)

        # Create model and compile
        model = Model(inputs=lr_input, outputs=hr_output)
        # model.summary()
        return model

In [ ]:
def build_discriminator():
    """
    Create a discriminator network using the hyperparameter values defined below
    :return:
    """
    leakyrelu_alpha = 0.2
    momentum = 0.8
    input_shape = (200, 200, 3)

    input_layer = Input(shape=input_shape)

    # Add the first convolution block
    dis1 = Conv2D(filters=64, kernel_size=3, strides=1, padding='same')(input_layer)
    dis1 = LeakyReLU(alpha=leakyrelu_alpha)(dis1)

    # Add the 2nd convolution block
    dis2 = Conv2D(filters=64, kernel_size=3, strides=2, padding='same')(dis1)
    dis2 = LeakyReLU(alpha=leakyrelu_alpha)(dis2)
    dis2 = BatchNormalization(momentum=momentum)(dis2)

    # Add the third convolution block
    dis3 = Conv2D(filters=128, kernel_size=3, strides=1, padding='same')(dis2)
    dis3 = LeakyReLU(alpha=leakyrelu_alpha)(dis3)
    dis3 = BatchNormalization(momentum=momentum)(dis3)

    # Add the fourth convolution block
    dis4 = Conv2D(filters=128, kernel_size=3, strides=2, padding='same')(dis3)
    dis4 = LeakyReLU(alpha=leakyrelu_alpha)(dis4)
    dis4 = BatchNormalization(momentum=0.8)(dis4)

    # Add the fifth convolution block
    dis5 = Conv2D(256, kernel_size=3, strides=1, padding='same')(dis4)
    dis5 = LeakyReLU(alpha=leakyrelu_alpha)(dis5)
    dis5 = BatchNormalization(momentum=momentum)(dis5)

    # Add the sixth convolution block
    dis6 = Conv2D(filters=256, kernel_size=3, strides=2, padding='same')(dis5)
    dis6 = LeakyReLU(alpha=leakyrelu_alpha)(dis6)
    dis6 = BatchNormalization(momentum=momentum)(dis6)


    # Add a dense layer
    dis7 = Dense(units=1024)(dis6)
    dis7 = LeakyReLU(alpha=0.2)(dis7)

    # Last dense layer - for classification
    output = Dense(units=1, activation='sigmoid')(dis7)

    model = Model(inputs=[input_layer], outputs=[output], name='discriminator')
    return model

In [ ]:
def build_vgg():
    """
    Build VGG network to extract image features
    """
    input_shape = (200, 200, 3)

    # Load a pre-trained VGG19 model trained on 'Imagenet' dataset
    # Ensure you have an active internet connection
    vgg = VGG19(weights="imagenet", include_top=False)
    vgg.outputs = [vgg.layers[9].output]

    input_layer = Input(shape=input_shape)

    # Extract features
    features = vgg(input_layer)

    # Create a Keras model
    model = Model(inputs=[input_layer], outputs=[features])
    return model

In [ ]:
if __name__ == '__main__':
    data_dir='/content/drive/My Drive/Super_Resolution/EDSR_ESRGAN/'
    epochs =600
    batch_size =1
    mode = 'train'

    # Shape of low-resolution and high-resolution images
    low_resolution_shape = (100, 100, 3)
    high_resolution_shape = (200, 200, 3)

    # Common optimizer for all networks
    common_optimizer = Adam(0.0002, 0.5)

    if mode == 'train':
        # Build and compile VGG19 network to extract features
        vgg = build_vgg()
        vgg.trainable = False
        vgg.compile(loss='mse', optimizer=common_optimizer, metrics=['accuracy'])

        # Build the EDSR network
        model = build_model()

        """
        Build and compile the final Network with content and perception loss
        """

        # Input layers for high-resolution and low-resolution images
        input_high_resolution = Input(shape=high_resolution_shape)
        input_low_resolution = Input(shape=low_resolution_shape)

        # Generate high-resolution images from low-resolution images
        generated_high_resolution_images = model(input_low_resolution)

        # Extract feature maps of the generated images
        features = vgg(generated_high_resolution_images)

        # Create and compile an adversarial model
        final_model = Model([input_low_resolution, input_high_resolution], [generated_high_resolution_images, features])
        final_model.compile(loss=['mean_absolute_error','mean_absolute_error'], loss_weights=[1e-3, 1], optimizer=common_optimizer)

        # Getting the training images

        train_hr , train_lr , no_images = dataPreProcessing((200,200),(100,100))

        # No. of iterations for an epoch
        noIter=int(no_images/batch_size)

        loss_array=[]

        for epoch in range(epochs):
            print("Epoch:{}".format(epoch))
            modelLoss=0

            """
            Train the discriminator network
            """

            for i in range(noIter):
              print("Epoch:{} Iteration:{}".format(epoch,i+1))


              # Sample a batch of images
              high_resolution_images , low_resolution_images = sampleImages(batch_size,train_hr,train_lr,no_images)


              #print("High Images: ",high_resolution_images)

              # Normalize images
              high_resolution_images = high_resolution_images / 255
              low_resolution_images = low_resolution_images / 255

              # Generate high-resolution images from low-resolution images
              generated_high_resolution_images = model.predict(low_resolution_images)

              # Extract feature maps for real high-resolution images
              image_features = vgg.predict(high_resolution_images)

              # Train the generator network
              loss = final_model.train_on_batch([low_resolution_images, high_resolution_images],
                                              [high_resolution_images, image_features])

              print("Loss:", loss)
              modelLoss+=loss[0]


            loss_array.append(modelLoss)
            if epoch%10 == 0:
              generated=[]

              for img in train_lr:
                batch_lr=np.array([img])/255

                generated_image = model.predict_on_batch(batch_lr)
                generated.append(generated_image[0])
              generated=np.array(generated)

              #stiched_hr=stichImages(train_hr,(2000,2000))
              #stiched_lr=stichImages(train_lr, (1000,1000))

              stiched_hr=stichImages(train_hr,(2000,2000))
              stiched_lr=stichImages(train_lr, (1000,1000))
              stiched_generated=stichImages(generated,(2000,2000))

              #stiched_generated=stichImages(generated,(2000,2000))
              #stiched_generated = 255*stiched_generated
              #stiched_generated =stiched_generated.astype('uint8')

              savepath=os.path.join(data_dir,'output1')

              #save_images(stiched_lr/255, stiched_hr/255, stiched_generated, path=os.path.join(savepath,'Epoch-{}'.format(epoch+1)))
              imsave(os.path.join(savepath,'Epoch-{}_FullImage.jpg'.format(epoch+1)),stiched_generated)


80134624/80134624 [==============================] - 0s 0us/step


IndexError: Exception encountered when calling layer "vgg19" (type Functional).

pop from empty list

Call arguments received by layer "vgg19" (type Functional):
  • inputs=tf.Tensor(shape=(None, 200, 200, 3), dtype=float32)
  • training=None
  • mask=None

In [ ]:
  epoch_array=[i for i in range(epochs)]
plot = plt.plot(epoch_array , loss_array)
plt.xlabel('Epoch')
plt.ylabel('Model Loss')
plt.title("Model Loss VS Epochs")
plt.savefig(os.path.join(data_dir , 'ESRGAN_EDSR_Loss_Epoch.jpg'))
plt.show()


NameError: name 'loss_array' is not defined

In [ ]:
modelpath=os.path.join(data_dir,'Saved_Models1')
model.save(os.path.join(modelpath,'ESRGAN_EDSR_SR.h5'))

NameError: name 'model' is not defined

In [ ]:
generated=[]

for img in train_lr:
  batch_lr=np.array([img])/255

  generated_image = model.predict_on_batch(batch_lr)
  generated.append(generated_image[0])
generated=np.array(generated)

#stiched_hr=stichImages(train_hr,(2000,2000))
#stiched_lr=stichImages(train_lr, (1000,1000))
stiched_generated=stichImages(generated,(2000,2000))
stiched_generated = 255*stiched_generated
stiched_generated =stiched_generated.astype('uint8')

savepath=os.path.join(data_dir,'Output_Images1')

save_images(stiched_lr/255, stiched_hr/255, stiched_generated, path=os.path.join(savepath,'Epoch-{}'.format(150)))
imsave(os.path.join(savepath,'Epoch-{}_FullImage.jpg'.format(150)),stiched_generated)

NameError: name 'train_lr' is not defined

In [ ]:
testImage=cv2.imread(os.path.join(data_dir , 'Test.jpg'))
test_lr , no_images= splitImg(testImage,(100,100))

generated=[]

for img in test_lr:
  batch_lr=np.array([img])/255

  generated_image = generator.predict_on_batch(batch_lr)
  generated.append(generated_image[0])
generated=np.array(generated)

stiched_generated=stichImages(generated,(2000,2000))
stiched_generated = 255 *stiched_generated # Now scale by 255
stiched_generated = stiched_generated.astype(np.uint8)

imsave(os.path.join(data_dir,'Test_FullImage.jpg'),stiched_generated)

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
%cd
with open('/content/drive/My Drive/Super_Resolution/SRGAN/tex.txt', 'w') as f:
    for i in loss_array:
        f.write(str(i)+',')

/root


NameError: name 'loss_array' is not defined

In [ ]:
glob.glob( os.path.join(path,'Train_Images1', '*.jpg'))

['/content/drive/My Drive/Super_Resolution/EDSR_ESRGAN/Train_Images1/cart2000.jpg']